# Mean OptiTrack workspace outline from 5 runs

This notebook uses the already-filtered outputs from `optitrack_position_analysis.ipynb` and builds a mean workspace (WS) outline from the 5 small-motion runs.

Coordinate assumption:
- **East (E)** = +X
- **West (W)** = -X
- **North (N)** = +Z
- **South (S)** = -Z
- Y is treated as the vertical / out-of-plane coordinate for the 3D outline.

The movement was performed on 16 axes, so this notebook extracts reach on 16 compass directions every 22.5? and then creates a smooth mean outline.

In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

try:
    from scipy.interpolate import CubicSpline
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False
    warnings.warn('SciPy unavailable; using linear interpolation instead of periodic cubic smoothing.')

RESULTS_DIR = Path('OT-results')
SUMMARY_CSV = RESULTS_DIR / 'position_summary.csv'
FILTERED_DIR = RESULTS_DIR / 'filtered_data'
OUTLINE_DIR = RESULTS_DIR / 'workspace_outline'
OUTLINE_DIR.mkdir(parents=True, exist_ok=True)

AXIS_COUNT = 16
ANGLES_DEG = np.arange(0, 360, 360 / AXIS_COUNT, dtype=float)  # 0=E, 90=N
COMPASS = ['E', 'ENE', 'NE', 'NNE', 'N', 'NNW', 'NW', 'WNW', 'W', 'WSW', 'SW', 'SSW', 'S', 'SSE', 'SE', 'ESE']

plt.rcParams['figure.dpi'] = 130
plt.rcParams['savefig.dpi'] = 180
plt.rcParams['axes.grid'] = True

In [ ]:
def load_small_run_files():
    summary = pd.read_csv(SUMMARY_CSV)
    small = summary[summary['motion_scale'].eq('small')].copy()
    small = small.sort_values('file').reset_index(drop=True)
    if len(small) < 5:
        raise ValueError(f'Expected 5 small runs, found only {len(small)}')
    if len(small) > 5:
        warnings.warn(f'Found {len(small)} small runs; using the first 5 sorted by file name.')
        small = small.iloc[:5].copy()
    return small


def axis_reach_for_run(filtered_csv):
    df = pd.read_csv(filtered_csv)
    x = df['filtered_X_mm'].to_numpy(dtype=float)
    y = df['filtered_Y_mm'].to_numpy(dtype=float)
    z = df['filtered_Z_mm'].to_numpy(dtype=float)
    t = df['time_s'].to_numpy(dtype=float)

    rows = []
    for angle_deg, compass in zip(ANGLES_DEG, COMPASS):
        theta = np.deg2rad(angle_deg)
        ux, uz = np.cos(theta), np.sin(theta)
        projection = x * ux + z * uz
        i = int(np.nanargmax(projection))
        reach = float(projection[i])
        rows.append({
            'angle_deg_E0_CCW': angle_deg,
            'compass': compass,
            'reach_2d_mm': reach,
            'axis_x_mm': reach * ux,
            'axis_z_mm': reach * uz,
            'actual_x_at_reach_mm': float(x[i]),
            'actual_y_at_reach_mm': float(y[i]),
            'actual_z_at_reach_mm': float(z[i]),
            'time_at_reach_s': float(t[i]),
            'perpendicular_error_mm': float(abs(-x[i] * uz + z[i] * ux)),
        })
    return pd.DataFrame(rows), df


def periodic_smooth(theta_deg, values, samples=721):
    theta = np.deg2rad(np.r_[theta_deg, 360.0])
    y = np.r_[values, values[0]]
    theta_fine = np.linspace(0, 2 * np.pi, samples)
    if SCIPY_AVAILABLE:
        cs = CubicSpline(theta, y, bc_type='periodic')
        return theta_fine, cs(theta_fine)
    return theta_fine, np.interp(theta_fine, theta, y)


def polygon_area_mm2(x, z):
    return 0.5 * abs(np.dot(x, np.roll(z, -1)) - np.dot(z, np.roll(x, -1)))


def curve_perimeter_mm(x, z):
    return float(np.sum(np.hypot(np.diff(np.r_[x, x[0]]), np.diff(np.r_[z, z[0]]))))

In [ ]:
small_runs = load_small_run_files()
all_axis_rows = []
run_traces = []

for _, row in small_runs.iterrows():
    filtered_csv = Path(row['filtered_csv'])
    if not filtered_csv.exists():
        filtered_csv = FILTERED_DIR / f"{Path(row['file']).stem.replace(' ', '_').replace('.', '.')}_filtered_position_mm.csv"
    axis_df, trace_df = axis_reach_for_run(filtered_csv)
    axis_df.insert(0, 'file', row['file'])
    all_axis_rows.append(axis_df)
    run_traces.append((row['file'], trace_df, axis_df))

axis_all = pd.concat(all_axis_rows, ignore_index=True)
axis_summary = (
    axis_all
    .groupby(['angle_deg_E0_CCW', 'compass'], as_index=False)
    .agg(
        mean_reach_2d_mm=('reach_2d_mm', 'mean'),
        std_reach_2d_mm=('reach_2d_mm', 'std'),
        min_reach_2d_mm=('reach_2d_mm', 'min'),
        max_reach_2d_mm=('reach_2d_mm', 'max'),
        mean_axis_x_mm=('axis_x_mm', 'mean'),
        mean_axis_z_mm=('axis_z_mm', 'mean'),
        mean_y_at_reach_mm=('actual_y_at_reach_mm', 'mean'),
        std_y_at_reach_mm=('actual_y_at_reach_mm', 'std'),
        mean_perpendicular_error_mm=('perpendicular_error_mm', 'mean'),
    )
    .sort_values('angle_deg_E0_CCW')
)

axis_all.to_csv(OUTLINE_DIR / 'five_small_runs_16axis_reach_each_run.csv', index=False)
axis_summary.to_csv(OUTLINE_DIR / 'five_small_runs_16axis_reach_mean.csv', index=False)

# Cardinal metrics from 16-axis mean reach.
cardinal = axis_summary.set_index('compass')
E = float(cardinal.loc['E', 'mean_reach_2d_mm'])
N = float(cardinal.loc['N', 'mean_reach_2d_mm'])
W = float(cardinal.loc['W', 'mean_reach_2d_mm'])
S = float(cardinal.loc['S', 'mean_reach_2d_mm'])
EW_span = E + W
NS_span = N + S
center_offset_x = (E - W) / 2.0
center_offset_z = (N - S) / 2.0

# Smooth mean outline.
theta_fine, reach_fine = periodic_smooth(axis_summary['angle_deg_E0_CCW'].to_numpy(), axis_summary['mean_reach_2d_mm'].to_numpy())
x_fine = reach_fine * np.cos(theta_fine)
z_fine = reach_fine * np.sin(theta_fine)
area_smooth = polygon_area_mm2(x_fine, z_fine)
perim_smooth = curve_perimeter_mm(x_fine, z_fine)

# Smooth +/- SD envelopes.
std = axis_summary['std_reach_2d_mm'].fillna(0).to_numpy()
_, upper_reach = periodic_smooth(axis_summary['angle_deg_E0_CCW'].to_numpy(), axis_summary['mean_reach_2d_mm'].to_numpy() + std)
_, lower_reach = periodic_smooth(axis_summary['angle_deg_E0_CCW'].to_numpy(), np.maximum(0, axis_summary['mean_reach_2d_mm'].to_numpy() - std))
x_upper, z_upper = upper_reach * np.cos(theta_fine), upper_reach * np.sin(theta_fine)
x_lower, z_lower = lower_reach * np.cos(theta_fine), lower_reach * np.sin(theta_fine)

metrics = {
    'runs_used': len(small_runs),
    'east_reach_mm': E,
    'west_reach_mm': W,
    'north_reach_mm': N,
    'south_reach_mm': S,
    'east_west_span_mm': EW_span,
    'north_south_span_mm': NS_span,
    'center_offset_x_mm_positive_east': center_offset_x,
    'center_offset_z_mm_positive_north': center_offset_z,
    'smooth_outline_area_mm2': area_smooth,
    'smooth_outline_perimeter_mm': perim_smooth,
    'max_mean_axis_reach_mm': float(axis_summary['mean_reach_2d_mm'].max()),
    'max_mean_axis_direction': str(axis_summary.loc[axis_summary['mean_reach_2d_mm'].idxmax(), 'compass']),
    'min_mean_axis_reach_mm': float(axis_summary['mean_reach_2d_mm'].min()),
    'min_mean_axis_direction': str(axis_summary.loc[axis_summary['mean_reach_2d_mm'].idxmin(), 'compass']),
    'mean_y_at_reach_range_mm': float(axis_summary['mean_y_at_reach_mm'].max() - axis_summary['mean_y_at_reach_mm'].min()),
}
pd.DataFrame([metrics]).to_csv(OUTLINE_DIR / 'five_small_runs_workspace_metrics.csv', index=False)

print('Runs used:')
for f in small_runs['file']:
    print(' -', f)
print('\nCardinal reach from mean 16-axis outline:')
print(f'  N (+Z): {N:.2f} mm')
print(f'  E (+X): {E:.2f} mm')
print(f'  S (-Z): {S:.2f} mm')
print(f'  W (-X): {W:.2f} mm')
print(f'  E-W span: {EW_span:.2f} mm')
print(f'  N-S span: {NS_span:.2f} mm')
print(f'  Smooth area: {area_smooth:.1f} mm^2')
print(f'  Smooth perimeter: {perim_smooth:.1f} mm')
print(f'  Largest mean direction: {metrics["max_mean_axis_direction"]} ({metrics["max_mean_axis_reach_mm"]:.2f} mm)')
axis_summary

In [ ]:
# 2D outline plot: individual 5 runs + mean smooth outline.
fig, ax = plt.subplots(figsize=(8.5, 8.5))

for file, _, axis_df in run_traces:
    theta = np.deg2rad(np.r_[axis_df['angle_deg_E0_CCW'].to_numpy(), 360.0])
    r = np.r_[axis_df['reach_2d_mm'].to_numpy(), axis_df['reach_2d_mm'].to_numpy()[0]]
    ax.plot(r * np.cos(theta), r * np.sin(theta), color='0.72', linewidth=1, alpha=0.85)

# Uncertainty envelope as +/- 1 SD.
ax.fill(np.r_[x_upper, x_lower[::-1]], np.r_[z_upper, z_lower[::-1]], color='tab:blue', alpha=0.13, label='mean +/- 1 SD')
ax.plot(x_fine, z_fine, color='tab:blue', linewidth=2.8, label='smooth mean outline')
ax.scatter(axis_summary['mean_axis_x_mm'], axis_summary['mean_axis_z_mm'], c='tab:blue', s=35, zorder=3, label='mean 16-axis points')

# Cardinal axes.
ax.axhline(0, color='0.25', linewidth=1)
ax.axvline(0, color='0.25', linewidth=1)
ax.annotate(f'N {N:.1f} mm', xy=(0, N), xytext=(0, N + 0.7), ha='center', va='bottom', color='tab:green', arrowprops=dict(arrowstyle='->', color='tab:green'))
ax.annotate(f'E {E:.1f} mm', xy=(E, 0), xytext=(E + 0.6, 0), ha='left', va='center', color='tab:orange', arrowprops=dict(arrowstyle='->', color='tab:orange'))
ax.annotate(f'S {S:.1f} mm', xy=(0, -S), xytext=(0, -S - 0.7), ha='center', va='top', color='tab:red', arrowprops=dict(arrowstyle='->', color='tab:red'))
ax.annotate(f'W {W:.1f} mm', xy=(-W, 0), xytext=(-W - 0.6, 0), ha='right', va='center', color='tab:purple', arrowprops=dict(arrowstyle='->', color='tab:purple'))

for _, row in axis_summary.iterrows():
    ax.text(row['mean_axis_x_mm'] * 1.08, row['mean_axis_z_mm'] * 1.08, row['compass'], fontsize=8, ha='center', va='center')

ax.set_aspect('equal', adjustable='box')
ax.set_xlabel('X displacement / East-West (mm)')
ax.set_ylabel('Z displacement / North-South (mm)')
ax.set_title('Mean workspace outline from 5 small OptiTrack runs\n16-axis smooth X-Z outline')
ax.legend(loc='best')
fig.tight_layout()
out_2d = OUTLINE_DIR / 'five_small_runs_mean_workspace_outline_2d.png'
fig.savefig(out_2d, bbox_inches='tight')
plt.close(fig)
print(out_2d)

In [ ]:
# 3D mean outline: X/Z radial reach with Y at the time each axis was reached.
y_mean = axis_summary['mean_y_at_reach_mm'].to_numpy()
theta_y, y_fine = periodic_smooth(axis_summary['angle_deg_E0_CCW'].to_numpy(), y_mean)

fig = plt.figure(figsize=(9.5, 8))
ax = fig.add_subplot(111, projection='3d')

# Individual run endpoint rings.
for file, _, axis_df in run_traces:
    theta = np.deg2rad(np.r_[axis_df['angle_deg_E0_CCW'].to_numpy(), 360.0])
    r = np.r_[axis_df['reach_2d_mm'].to_numpy(), axis_df['reach_2d_mm'].to_numpy()[0]]
    y = np.r_[axis_df['actual_y_at_reach_mm'].to_numpy(), axis_df['actual_y_at_reach_mm'].to_numpy()[0]]
    ax.plot(r * np.cos(theta), y, r * np.sin(theta), color='0.72', linewidth=0.9, alpha=0.7)

ax.plot(x_fine, y_fine, z_fine, color='tab:blue', linewidth=3, label='smooth mean 3D outline')
ax.scatter(axis_summary['mean_axis_x_mm'], axis_summary['mean_y_at_reach_mm'], axis_summary['mean_axis_z_mm'], c='tab:blue', s=35)

# X-Z projection/shadow at min Y for readability.
y_floor = min(float(y_fine.min()), float(axis_summary['mean_y_at_reach_mm'].min())) - 0.6
ax.plot(x_fine, np.full_like(x_fine, y_floor), z_fine, color='tab:blue', linewidth=1.2, alpha=0.35, label='X-Z projection')

for _, row in axis_summary.iterrows():
    ax.text(row['mean_axis_x_mm'] * 1.04, row['mean_y_at_reach_mm'], row['mean_axis_z_mm'] * 1.04, row['compass'], fontsize=7)

ax.set_xlabel('X / East-West (mm)')
ax.set_ylabel('Y vertical/out-of-plane (mm)')
ax.set_zlabel('Z / North-South (mm)')
ax.set_title('Mean workspace outline from 5 runs - 3D endpoint curve')
ax.legend(loc='upper left')
ax.view_init(elev=22, azim=-55)
fig.tight_layout()
out_3d = OUTLINE_DIR / 'five_small_runs_mean_workspace_outline_3d.png'
fig.savefig(out_3d, bbox_inches='tight')
plt.close(fig)
print(out_3d)

In [ ]:
# Save a compact human-readable interpretation.
shape_note = 'south-biased / vertically elongated in the X-Z plane' if S > N and NS_span >= EW_span else 'roughly balanced / check axes'
md = f"""# Mean 5-run workspace outline

Coordinate assumption: +X=East, -X=West, +Z=North, -Z=South. Y is vertical/out-of-plane.

## Runs used
{chr(10).join('- ' + f for f in small_runs['file'])}

## Mean cardinal reach
- North (+Z): **{N:.2f} mm**
- East (+X): **{E:.2f} mm**
- South (-Z): **{S:.2f} mm**
- West (-X): **{W:.2f} mm**
- East-West span: **{EW_span:.2f} mm**
- North-South span: **{NS_span:.2f} mm**

## Shape metrics
- Smooth 2D outline area: **{area_smooth:.1f} mm^2**
- Smooth 2D perimeter: **{perim_smooth:.1f} mm**
- Largest mean 16-axis reach: **{metrics['max_mean_axis_direction']}**, {metrics['max_mean_axis_reach_mm']:.2f} mm
- Smallest mean 16-axis reach: **{metrics['min_mean_axis_direction']}**, {metrics['min_mean_axis_reach_mm']:.2f} mm
- Center offset relative to start-origin: X={center_offset_x:+.2f} mm, Z={center_offset_z:+.2f} mm
- Mean Y variation across reached 16-axis endpoints: **{metrics['mean_y_at_reach_range_mm']:.2f} mm**

## Interpretation
The averaged outline is **{shape_note}**. South reach is larger than North, while East/West are closer to balanced. The useful final representation is the smooth 16-axis X-Z outline; the 3D curve adds the small Y/out-of-plane component at the reached points.

## Output files
- `five_small_runs_mean_workspace_outline_2d.png`
- `five_small_runs_mean_workspace_outline_3d.png`
- `five_small_runs_16axis_reach_mean.csv`
- `five_small_runs_16axis_reach_each_run.csv`
- `five_small_runs_workspace_metrics.csv`
"""
(OUTLINE_DIR / 'five_small_runs_workspace_interpretation.md').write_text(md, encoding='utf-8')
print(md)